In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama pull mistral
!pip install pdfplumber
!pip install langchain langchain-community


In [ ]:
import subprocess
import time

# Start Ollama server in the background
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server some time to start
time.sleep(5)

# Check if the server is running
try:
    response = subprocess.run(["curl", "http://localhost:11434"], capture_output=True, text=True)
    print("Ollama server response:", response.stdout)
except Exception as e:
    print("Failed to connect to Ollama server:", e)

In [ ]:
import pandas as pd
import pdfplumber
import json
from langchain_community.llms import Ollama
from tqdm import tqdm # A library to show a progress bar, useful for long loops

# ========================
# Tools for reading data (Unchanged)
# ========================
def read_jobs_tool(_=""):
    df = pd.read_csv("job_offers.csv")
    # Return a list of dictionaries instead of a single text block
    job_list = []
    for _, row in df.iterrows():
        job_list.append({
            "title": row['title'],
            "url": row['url'],
            "description": row['description']
        })
    return job_list

def read_cv_tool(file_path="cv.pdf"):
    with pdfplumber.open(file_path) as pdf:
        text = ""
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

# ========================
# LLM Configuration (Unchanged)
# ========================
llm = Ollama(model="mistral") 

# ========================
# The New "One-by-One" Analysis Function
# ========================
def analyze_single_job(job_data, cv_content):
    """
    Asks the LLM to analyze a SINGLE job against the CV and return a JSON.
    """
    
    # Create a small, focused prompt for one job
    prompt = f"""
    Voici le CV d'un candidat :
    --- DEBUT DU CV ---
    {cv_content}
    --- FIN DU CV ---

    Voici une offre d'emploi :
    --- DEBUT DE L'OFFRE ---
    Titre : {job_data['title']}
    URL : {job_data['url']}
    Description : {job_data['description']}
    --- FIN DE L'OFFRE ---

    Analyse cette offre et le CV. Extrais les informations de l'offre et détermine un score de compatibilité.
    
    Réponds IMPERATIVEMENT et UNIQUEMENT dans le format JSON suivant. Trouve la localisation dans la description.
    {{
        "title": "{job_data['title']}",
        "company": "string (le nom de l'entreprise)",
        "location": "string (la ville et le pays, ex: 'Paris, France')",
        "skills": ["string", "string"],
        "url": "{job_data['url']}",
        "match_score": "integer (0-100)"
    }}
    """
    
    try:
        response = llm.invoke(prompt)
        # Clean up the response in case the model adds markdown backticks
        cleaned_response = response.strip().replace("```json", "").replace("```", "")
        return json.loads(cleaned_response)
    except (json.JSONDecodeError, TypeError):
        # If the LLM fails to produce valid JSON, return None
        return None



In [ ]:
# ========================
# Main Execution Logic
# ========================
print("Lecture des données...")
all_jobs = read_jobs_tool()
cv = read_cv_tool()
analyzed_jobs = []

print(f"Analyse de {len(all_jobs)} offres d'emploi (cela peut prendre du temps)...")
# Loop through each job and analyze it
for job in tqdm(all_jobs):
    result = analyze_single_job(job, cv)
    print(result)
    if result: # Only add if the analysis was successful
        analyzed_jobs.append(result)

print("\nAnalyse terminée. Tri des résultats...")



In [ ]:
def safe_int(value):
    try:
        # Essaye de parser si c'est un nombre en string
        return int(value)
    except (ValueError, TypeError):
        # Si c'est "High (....)" ou autre texte
        # on peut aussi mapper High/Medium/Low à un score
        if isinstance(value, str):
            val = value.lower()
            if "high" in val:
                return 90
            elif "medium" in val:
                return 60
            elif "low" in val:
                return 30
        return 0

sorted_jobs = sorted(analyzed_jobs, key=lambda x: safe_int(x.get('match_score', 0)), reverse=True)

final_output = {
    "jobs": sorted_jobs[:10]
}

print("\n--- TOP 10 COMPATIBILITÉ ---")
print(json.dumps(final_output, indent=2, ensure_ascii=False))


In [ ]:
import pandas as pd

# Convertir la liste des jobs en DataFrame
df_top10 = pd.DataFrame(final_output["jobs"])

# Sauvegarder en CSV
df_top10.to_csv("top10_jobs.csv", index=False, encoding="utf-8-sig")

print("✅ Fichier 'top10_jobs.csv' sauvegardé avec succès !")
